# AutoFloods tutorial: one tile, one month, on a laptop

This notebook runs the full AutoFloods pipeline end to end for a single
Bihar grid tile (ID 318) with a narrowed wet-season window (August 2024
only, instead of the full July–October window used in the paper's Bihar
results), so it completes in a few minutes on a laptop rather than the
~7–12 minutes a full tile-year normally takes.

**Requirements:**
- `pip install -e .` from the repo root (or the dependencies in `pyproject.toml`)
- Internet access to NASA's CMR/ASF STAC endpoint (OPERA RTC-S1 data)
- A NASA Earthdata login configured in `~/.netrc` (free, public data access
  only — see https://urs.earthdata.nasa.gov/)
- No HPC, no SLURM, no GPU

Measured runtime on this exact configuration: **~4.3 minutes** (260.5s),
on an 8-core machine with a typical broadband connection.

In [ ]:
import sys
sys.path.append('..')  # run from notebooks/, package root one level up

from autofloods import flood_mapper
from autofloods.sources import OPERASource
from autofloods.detectors import ZScoreDetector

## 1. Configure the flood mapper

One tile (318, part of the Bihar grid), one dry season (April–May 2024,
used to fit the Z-score baseline), and a narrowed wet season (August 2024
only) so the whole run stays short. A full production run widens
`wet_duration` to `['2024/07', '2024/10']` — see Section 2.3 of the paper
and `examples/bihar_2024_opera_config.yaml` for the exact production
configuration.

In [ ]:
fm = flood_mapper(
    grid_shapefile='../resources/india_utm_fishnet_buffer.gpkg',
    grid_id_list=[318],
    dry_years=[2024, 2024],
    wet_duration=['2024/08', '2024/08'],   # narrowed to 1 month for tutorial speed
    slope_dir='../resources/slope',
    source=OPERASource(),                   # swap for MPCSource() to use the Azure/MPC backend instead
    detector=ZScoreDetector(vv_thd=-2.5, vh_thd=-2.5),
    output_dir='../output/tutorial_tile318',
    cell_size=30,
)

## 2. Dry-season baseline

Searches, downloads, and reads the dry-season Sentinel-1 scenes, then
fits the per-pixel VV/VH mean and standard deviation used as the Z-score
baseline. This is the slowest step (~2–3 minutes) and is cached to disk
— rerunning this notebook a second time will skip it entirely.

In [ ]:
fm.get_dry_dates()
fm.generate_dry_date_ranges()
fm.get_s1_items(dry_wet='dry')
print(f'dry scenes found: {len(fm.dry_aoi_scene_dict.get(318, []))}')
fm.read_scenes(dry_wet='dry')
fm.generate_mean_std_by_aoi()

## 3. Slope mask

Downloads (or reuses, if already cached in `slope_dir`) a DEM-derived
slope layer for this tile, used to suppress false positives on steep
terrain.

In [ ]:
fm.prepare_slope()

## 4. Wet-season detection

Reads the (narrowed) wet-season scenes, computes the Z-score flood
classification against the dry-season baseline, applies the slope mask,
merges same-date scenes, and aggregates into a monthly flood-day-count
raster — the same final product format used throughout the paper.

In [ ]:
fm.prepare_wet_scenes()
print(f'wet scenes found: {sum(len(v) for v in fm.wet_scenes_by_aoi.values())}')
fm.map_floods()
fm.merge_floods_by_date(export_raster=True)
fm.generate_number_of_scenes(export_raster=True)
fm.monthly_sum()

## 5. Look at the result

The monthly output is a Cloud Optimized GeoTIFF at
`../output/tutorial_tile318/flood_raster/monthlyadded.../..._monthly.tif`,
one band (here, just one: August 2024), pixel value = count of
flooded observations that month.

In [ ]:
import glob
import rasterio
import matplotlib.pyplot as plt

monthly_tif = glob.glob('../output/tutorial_tile318/flood_raster/monthlyadded*/*_monthly.tif')[0]
with rasterio.open(monthly_tif) as src:
    band = src.read(1, masked=True)

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(band, cmap='Blues', vmin=0)
ax.set_title('Tile 318, August 2024 — flooded observations')
ax.axis('off')
fig.colorbar(im, ax=ax, shrink=0.7, label='flooded observation count')
plt.show()

## Next steps

- Widen `wet_duration` back to `['2024/07', '2024/10']` for the full
  production window (takes ~7–12 minutes instead of ~4).
- Add more tile IDs to `grid_id_list` to process a larger area.
- See `examples/bihar_2024_opera_config.yaml` and
  `scripts/run_autofloods.py` for the config-driven CLI used for the
  full 198-tile-year Bihar reprocess in the paper.